In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

# --- Load Data and Create a Unified DataFrame ---
PKL_PATH = '../../data/main_figures/slr_res/DIVA_data/Coast_with_Mang.pkl' 
COASTLINE_DATA_PATH = "../../data/main_figures/slr_res/DIVA_coastline_WGS84_Point.csv"
with open(PKL_PATH, 'rb') as f:
    data = pickle.load(f)

# # Extract relevant data
df = pd.DataFrame({
    'ClsFID': data['ClsFID'],
    'area': data['area'],
    'agb_present': data['agb_present'],
    'agb_grow': data['agb_grow'],
    'agb_126': data['agb_126'],
    'agb_585': data['agb_585'],
    # 注意：根据您的脚本，这里选择的是第2列(Pop5)和第1列(50%恢复)
    'area_after_SLR_126': data['area_after_SLR_126'][:, 1], 
    'area_after_SLR_585': data['area_after_SLR_585'][:, 1],
    'area_restoration_126': data['area_restoration_126'][:, 0],
    'area_restoration_585': data['area_restoration_585'][:, 0],
})

# --- Load and Merge Country Information ---
# 从新的基础数据文件中读取国家信息
coastline_data = pd.read_csv(COASTLINE_DATA_PATH)
# 使用.merge()方法，通过'ClsFID'将国家信息精确地匹配到每一行
# Parse the 'ID_Country' column to extract the clean country name.
# Split the column on the first space. expand=True creates new columns.
country_split = coastline_data["ID_Country"].str.split(" ", n=1, expand=True)
# Create the country_info DataFrame for merging.
# We take the original CLSFID for merging and the newly extracted country name (column 1).
country_info = pd.DataFrame({'ClsFID': coastline_data['CLSFID'], 'Country': country_split[1]})
# 将主数据框与国家信息合并
df = pd.merge(df, country_info, on='ClsFID', how='left')

# --- Calculate AGB Changes (Vectorized Operations) ---
# 定义一个函数，使其在DataFrame上操作，更清晰
def calculate_agb_changes(df, scenario_suffix):
    agb_scen = f'agb_{scenario_suffix}'
    area_slr_scen = f'area_after_SLR_{scenario_suffix}'
    area_res_scen = f'area_restoration_{scenario_suffix}'
    # 直接在DataFrame上创建新列，单位转换为 Tg (10^6 Mg)
    df[f'total_change_{scenario_suffix}'] = (df[agb_scen] * (df[area_slr_scen] + df[area_res_scen]) - df['agb_present'] * df['area']) / 1e6
    df[f'grow_change_{scenario_suffix}'] = (df['agb_grow'] * df['area'] - df['agb_present'] * df['area']) / 1e6
    df[f'ht_change_{scenario_suffix}'] = (df[agb_scen] * df['area'] - df['agb_grow'] * df['area']) / 1e6
    df[f'res_change_{scenario_suffix}'] = (df[agb_scen] * df[area_res_scen]) / 1e6
    df[f'slr_change_{scenario_suffix}'] = (df[agb_scen] * (df[area_slr_scen] - df['area'])) / 1e6
    return df
df = calculate_agb_changes(df, '126')
df = calculate_agb_changes(df, '585')

# --- Aggregate Results by Country using groupby ---
agg_cols_126 = {
    'AGB_Change': 'total_change_126',
    'AGB_Change_Grow': 'grow_change_126',
    'AGB_Change_HT': 'ht_change_126',
    'AGB_Change_RES': 'res_change_126',
    'AGB_Change_SLR': 'slr_change_126'
}
agg_cols_585 = {
    'AGB_Change': 'total_change_585',
    'AGB_Change_Grow': 'grow_change_585',
    'AGB_Change_HT': 'ht_change_585',
    'AGB_Change_RES': 'res_change_585',
    'AGB_Change_SLR': 'slr_change_585'
}
# Invert the dictionaries for the rename function.
rename_map_126 = {v: k for k, v in agg_cols_126.items()}
rename_map_585 = {v: k for k, v in agg_cols_585.items()}

# 使用groupby().sum()一步完成按国家汇总，然后使用翻转后的字典进行重命名
country_agb_change_126 = df.groupby('Country', as_index=False)[list(agg_cols_126.values())].sum().rename(columns=rename_map_126).sort_values(by='AGB_Change')
country_agb_change_585 = df.groupby('Country', as_index=False)[list(agg_cols_585.values())].sum().rename(columns=rename_map_585).sort_values(by='AGB_Change')



In [ ]:
# Present-day country-level AGB totals used by the top-country panels.
df['agb_present_total'] = df['agb_present'] * df['area'] / 1e6
country_present_agb = df.groupby('Country', as_index=False)['agb_present_total'].sum()
country_analysis_126 = pd.merge(country_agb_change_126, country_present_agb, on='Country')
country_analysis_585 = pd.merge(country_agb_change_585, country_present_agb, on='Country')


In [ ]:
# --- 1. 确定AGB总量前十的国家 ---
top_10_countries_by_total_agb = country_analysis_126.sort_values(
    by='agb_present_total', ascending=False
).head(10)['Country'].tolist()

# --- 2. 准备用于绘图的数据 ---
# 按当前AGB总量降序排列 (从大到小)
data_for_plot_126 = country_analysis_126[
    country_analysis_126['Country'].isin(top_10_countries_by_total_agb)
].sort_values(by='agb_present_total', ascending=False)

data_for_plot_585 = country_analysis_585[
    country_analysis_585['Country'].isin(top_10_countries_by_total_agb)
].sort_values(by='agb_present_total', ascending=False)

# --- 3. 创建图表和子图 (2行1列，共享X轴) ---
fig_new, axs_new = plt.subplots(2, 1, figsize=(12, 12), sharex=True)
fig_new.supylabel('AGB Change (Tg DM)', fontsize=16)

# --- 定义一个可复用的绘图函数 ---
def plot_top10_bars(ax, df_data, title_char, scenario_text, show_legend=False):
    """一个函数来绘制每个子图，避免代码重复"""
    
    change_components = df_data[['AGB_Change_Grow', 'AGB_Change_HT', 'AGB_Change_RES', 'AGB_Change_SLR']]
    bar_colors = ['#76d7c4', '#f3b6b1', '#ebdef7', '#aad6f1']

    change_components.plot(kind='bar', stacked=True, ax=ax, color=bar_colors, edgecolor='none', width=0.7, legend=False)
    ax.scatter(range(len(df_data)), df_data['AGB_Change'], facecolor='none', edgecolor='black', s=100, zorder=5)
    ax.axhline(0, color='grey', linestyle='--', linewidth=1, zorder=0)

    ax.set_title(title_char, loc='left', fontsize=16, fontweight='bold')
    ax.text(0.02, 0.95, scenario_text, transform=ax.transAxes, ha='left', va='top', fontsize=16)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    ax.tick_params(axis='y')
    ax.set_xlabel('')
    
    if show_legend:
        bar_labels = ['Potential growth to maturity', 'Climate and hydrology change', 'Restoration', 'Sea-level rise']
        bar_handles = [plt.Rectangle((0,0),1,1, color=color) for color in bar_colors]
        scatter_handle = plt.Line2D([0], [0], marker='o', color='k', linestyle='None', markersize=8, fillstyle='none', label='Total')
        
        all_handles = [scatter_handle] + bar_handles
        all_labels = ['Total'] + bar_labels
        
        ax.legend(handles=all_handles, labels=all_labels,
                  loc='lower right', frameon=False, labelspacing=0.6)

# --- 绘制上下两个子图 ---
plot_top10_bars(axs_new[0], data_for_plot_126, 'a', 'Low-warming scenario', show_legend=False)
plot_top10_bars(axs_new[1], data_for_plot_585, 'b', 'High-warming scenario', show_legend=True)

# *** 修正：独立调整每个子图的Y轴范围 ***

# 调整上方子图 (Low-warming)
y1_min, y1_max = axs_new[0].get_ylim()
data_range_1 = y1_max - y1_min
adjusted_ymax_1 = y1_max + data_range_1 * 0.1
axs_new[0].set_ylim(bottom=y1_min, top=adjusted_ymax_1)

# 调整下方子图 (High-warming)
y2_min, y2_max = axs_new[1].get_ylim()
data_range_2 = y2_max - y2_min
adjusted_ymax_2 = y2_max + data_range_2 * 0.1
axs_new[1].set_ylim(bottom=y2_min, top=adjusted_ymax_2)


# --- 5. 设置共享的X轴 ---
plt.sca(axs_new[1])
plt.xticks(ticks=range(len(data_for_plot_585)), labels=data_for_plot_585['Country'], rotation=45, ha='right')
plt.tick_params(axis='x', length=5)

# --- 6. 最终布局调整 ---
plt.tight_layout()

# --- 保存并显示图像 ---
plt.savefig('../../figures/main_text/fig04_country_change.png', dpi=300, bbox_inches='tight')
plt.show()